# practice # 7
mulit agent구성

각각의 tool 을 가지는 agent 생성

query -> agent#1 -> agent#2 -> Agnet#1 반복 -> end

llm 성능과 prompt 에 따라 성능이 많이 좌우됨. 
현재 상태로는 프롬프트 개선이 메우 필요함
gpt 로는 수행 가능하나 gemini로는 수행이 어려울 정도

1. 프롬프트 개선
2. 가져온 데이터 split하여 필요데이터만 전달

## 구성도
![구성도](/home/ansgyqja/AI_application/images/7.practice.png)



In [14]:
import os,sys
sys.path.append(os.path.abspath(os.path.join(os.path.dirname('utils'), '..')))
from module.utils import * 
from module.prompt import * 
from module.custom_model import *
from typing import List,Annotated,TypedDict
from typing import TypedDict, Annotated, List, Literal,Tuple
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import AIMessage,HumanMessage,SystemMessage,ToolMessage
from langgraph.prebuilt import create_react_agent

In [15]:
start_langsmith('practice_7')

LangSmith 추적을 시작합니다.
[프로젝트명]
practice_7


In [16]:
from langchain_experimental.utilities import PythonREPL
from langchain_core.tools import tool

python_repl = PythonREPL()
# Python 코드를 실행하는 도구 정의
@tool
def python_repl_tool(
    code: Annotated[str, "The python code to execute to generate your chart."],
):
    """Use this to execute python code. If you want to see the output of a value,
    you should print it out with `print(...)`. This is visible to the user."""
    try:
        # 주어진 코드를 Python REPL에서 실행하고 결과 반환
        result = python_repl.run(code)
    except BaseException as e:
        return f"Failed to execute code. Error: {repr(e)}"
    # 실행 성공 시 결과와 함께 성공 메시지 반환
    result_str = f"Successfully executed:\n```python\n{code}\n```\nStdout: {result}"
    return (
        result_str + "\n\nIf you have completed all tasks, respond with FINAL ANSWER."
    )

# Chart Generator Agent 생성

add_prompt =  """
You can only generate charts of `python_repl_tool`. You are working with a researcher colleague.
Be sure to use the following font code in your code when generating charts.
"""

chart_prompt = get_prompt_multi_chart()

chart_agent = create_react_agent(
    # get_gpt(),
    get_gemini(),
    [python_repl_tool],
    prompt=chart_prompt
)

In [17]:
class State(TypedDict):
    messages:Annotated[list,add_messages]
    answer: Annotated[list,add_messages]
    sender : Annotated[str, 'the last send agent name']



In [18]:
web_search = get_tavily_tool()
web_prompt = get_prompt_multi_web()
web_agent = create_react_agent(
    # get_gpt(),
    get_gemini(),
    [web_search],
    prompt=web_prompt
    ) 

In [19]:
# web_agent.invoke({'messages':['2024년 서울 인구와 경기 인구 비교 그래프로 보여줘']})

In [20]:
def web_agent_node(state : State):
    messages = state['messages']
    result = web_agent.invoke({'messages':messages})
    latest_messages = HumanMessage(
        content=result["messages"][-1].content, name="chart_generator"
    )
    return State({'messages':[latest_messages]})

In [21]:
def chart_agent_node(state: State) -> State:
    result = chart_agent.invoke(state)

    # 마지막 메시지를 HumanMessage 로 변환
    last_message = HumanMessage(
        content=result["messages"][-1].content, name="chart_generator"
    )
    return {
        # share internal message history of chart agent with other agents
        "messages": [last_message],
    }

In [22]:
def router(state: State):
    # This is the router
    messages = state["messages"]
    last_message = messages[-1]
    if "FINAL ANSWER" in last_message.content:
        # Any agent decided the work is done
        return END
    return "continue"

In [23]:
state = StateGraph(State)
state.add_node("web_agent_node", web_agent_node)
state.add_node("chart_agent_node", chart_agent_node)

state.add_conditional_edges(
    "web_agent_node",
    router,
    {"continue": "chart_agent_node", END: END},
)
state.add_conditional_edges(
    "chart_agent_node",
    router,
    {"continue": "web_agent_node", END: END},
)

state.add_edge(START, "web_agent_node")
graph = state.compile(checkpointer=get_check_pointer())

In [24]:
# visualize_graph(graph)
print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	web_agent_node(web_agent_node)
	chart_agent_node(chart_agent_node)
	__end__([<p>__end__</p>]):::last
	__start__ --> web_agent_node;
	chart_agent_node -.-> __end__;
	chart_agent_node -. &nbsp;continue&nbsp; .-> web_agent_node;
	web_agent_node -.-> __end__;
	web_agent_node -. &nbsp;continue&nbsp; .-> chart_agent_node;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [25]:
uuid = get_random_uuid()
config = get_runnable_config(recursion_limit=5,thread_id=uuid)

In [26]:
inputs=  {'messages':['2024년 서울 남자와 여자 성비 알려줘']}
stream_graph(graph,inputs,config)


🔄 Node: web_agent_node 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
2024년 서울시의 성별 인구 비율에 대한 정보는 아직 제공되지 않았습니다. 관련 통계 자료가 발표되면 알려드리겠습니다.
🔄 Node: chart_agent_node 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
FINAL ANSWER: 2024년 서울시의 성별 인구 비율에 대한 정보는 아직 제공되지 않았습니다. 관련 통계 자료가 발표되면 알려드리겠습니다.